# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/
MemTotal: 1007.72 GB
MemFree: 338.63 GB
MemAvailable: 708.86 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successf

## 2. Analyze model dtypes

In [3]:
import torch
from transformers import AutoModelForCausalLM
from collections import defaultdict
import numpy as np

def analyze_model_dtypes(model_name: str):
    # Load model
    print(f"Loading model {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    
    # Initialize counters
    dtype_counts = defaultdict(int)
    total_params = 0
    
    # Analyze parameters
    print("\nAnalyzing parameter dtypes...")
    for name, param in model.named_parameters():
        num_params = param.numel()
        dtype_counts[param.dtype] += num_params
        total_params += num_params
        
        # Print details for each layer
        print(f"{name}: {param.dtype} (shape: {param.shape})")
    
    # Calculate and print statistics
    print("\nSummary:")
    print(f"Total parameters: {total_params:,}")
    for dtype, count in dtype_counts.items():
        percentage = (count / total_params) * 100
        print(f"{dtype}: {count:,} parameters ({percentage:.2f}%)")

if __name__ == "__main__":
    model_name = "meta-llama/Meta-Llama-3-8B"
    analyze_model_dtypes(model_name)

Loading model meta-llama/Meta-Llama-3-8B...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Analyzing parameter dtypes...
model.embed_tokens.weight: torch.bfloat16 (shape: torch.Size([128256, 4096]))
model.layers.0.self_attn.q_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.0.self_attn.k_proj.weight: torch.bfloat16 (shape: torch.Size([1024, 4096]))
model.layers.0.self_attn.v_proj.weight: torch.bfloat16 (shape: torch.Size([1024, 4096]))
model.layers.0.self_attn.o_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.0.mlp.gate_proj.weight: torch.bfloat16 (shape: torch.Size([14336, 4096]))
model.layers.0.mlp.up_proj.weight: torch.bfloat16 (shape: torch.Size([14336, 4096]))
model.layers.0.mlp.down_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 14336]))
model.layers.0.input_layernorm.weight: torch.bfloat16 (shape: torch.Size([4096]))
model.layers.0.post_attention_layernorm.weight: torch.bfloat16 (shape: torch.Size([4096]))
model.layers.1.self_attn.q_proj.weight: torch.bfloat16 (shape: torch.Size([4096, 4096]))
model.layers.1

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


In [ ]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)

sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


## 3. Loading Datasets

### 3.1. WikiText

In [4]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=3,
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622


Token indices sequence length is longer than the specified maximum sequence length for this model (252726 > 2048). Running this sequence through the model will result in indexing errors



Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 252726
Tokenizer compression rate: 22.13%

Number of batches in validation dataloader: 564

Batch 1:
  Original Text:  for people he did not respect. Scenes in Ricky's household reflect Ball's own childhood experiences. Ball suspected his father was homosexual and used the idea to create Col. Fitts, a man who " gave up his chance to be himself ". Ball said the script's mix of comedy and drama was not intentional, but that it came unconsciously from his own outlook on life. He said the juxtaposition produced a starker contrast, giving each trait more impact than if they appeared alone. 


 In the script that was...
  Input data (first 5 tokens): tensor([ 369, 1274,  568, 1550,  539])
  Target labels (first 5 tokens): tensor([-100, -100, -100, -100, -100])
  Input data shape: torch.Size([1, 2048])
  Target labels shape: torch.Size([1, 2048])


In [7]:
for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")


Batch 1:


In [ ]:
data

In [16]:
num_to_print = 5  # Define the number of elements to print

for i, batch in enumerate(wikitext_dataloader):
  if i >= num_to_print:
    break
  print(f"Batch {i+1}:")
  print(batch)  # This will print the data and labels (if applicable)


Batch 1:
[tensor([[29896, 29953,   732,  ...,   310, 10013, 22309]]), tensor([[ -100,  -100,  -100,  ...,  -100,  -100, 22309]])]
Batch 2:
[tensor([[13978,   287,   491,  ...,   471, 10624,   491]]), tensor([[-100, -100, -100,  ..., -100, -100,  491]])]
Batch 3:
[tensor([[26835, 23606,  1723,  ...,   278,  1736,   515]]), tensor([[-100, -100, -100,  ..., -100, -100,  515]])]
Batch 4:
[tensor([[ 297,  278, 4517,  ...,  505, 7371, 6625]]), tensor([[-100, -100, -100,  ..., -100, -100, 6625]])]
Batch 5:
[tensor([[17063,  1304,   363,  ...,   363,  4359,  1023]]), tensor([[-100, -100, -100,  ..., -100, -100, 1023]])]


### 3.2. OpenAssistant

In [4]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=512,
  tokenizer_name=model_name,
  seed=1
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {oasst_dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Length of datasets: 80215 4401
Number of batches in train_dataloader: 1063
Batch 1:
  Original Text:  of data to avoid overfitting. Also, care must be taken to choose appropriate activation functions and model architecture to ensure that the network is capable of accurately representing the data.






Highlighting the drawbacks:

Gaussian Mixture Model (GMM) is computationally expensive and sensitive to initialization, which means that it can converge to suboptimal solutions. It may also be difficult to determine the number of mixture components needed for a particular dataset.

Kernel Density
  Input data (first 5 tokens): tensor([ 315,  828,  311, 5766,  927])
  Target labels (first 5 tokens): tensor([ 828,  311, 5766,  927, 6410])
  Input data shape: torch.Size([1, 512])
  Target labels shape: torch.Size([1, 512])
Batch 2:
  Original Text:  должны обеспечивать кровоснабжение мозга на большой высоте и подъем крови от ног. Опорно-двигательная система должна выдерживать намного более 

### 3.3 C4

In [5]:
# Initialize the datamodule
import os
from src.data.C4DataModule import C4DataModule

print("\n################################")
print("Setting up C4DataModule...")
print("################################\n")

# c4_sequence_length = tokenizer.model_max_length
# c4_sequence_length = 10
c4_sequence_length = 2048
c4_batch_size = 1  # Just use batch size 1 for this project
c4_stride = 2048
c4_seed = 3
# c4_n_lines = 406
c4_n_lines = None

c4_data_module = C4DataModule(
  directory_dataset=os.getcwd(),
  batch_size=c4_batch_size,
  sequence_length=c4_sequence_length,
  stride=c4_stride,
  tokenizer_name=model_name,
  seed=c4_seed,
  n_lines = c4_n_lines
)

c4_dataloader = c4_data_module.test_dataloader()

print("\n################################")
print("Printing properties of C4DataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(c4_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(c4_data_module.val_dataset)}")
print(f"Length of test dataset: {len(c4_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in c4_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in c4_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in c4_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in c4_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(c4_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(c4_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up C4DataModule...
################################



Generating train split:   0%|          | 356317/364868892 [00:11<3:24:11, 29753.23 examples/s]


ExpectedMoreSplitsError: {'validation'}

### 3.4 PTB

In [ ]:
# Initialize the datamodule
import os
from src.data.C4DataModule import C4DataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

# wikitext_sequence_length = tokenizer.model_max_length
# wikitext_sequence_length = 10
c4_sequence_length = 2048
c4_batch_size = 1  # Just use batch size 1 for this project
c4_stride = 2048
c4_seed = 3
# wikitext_n_lines = 406
c4_n_lines = 1000

c4_data_module = C4DataModule(
  directory_dataset=os.getcwd(),
  batch_size=c4_batch_size,
  sequence_length=c4_sequence_length,
  stride=c4_stride,
  tokenizer_name=model_name,
  seed=c4_seed,
  n_lines = c4_n_lines
)

c4_dataloader = c4_data_module.test_dataloader()

print("\n################################")
print("Printing properties of C4DataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(c4_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(c4_data_module.val_dataset)}")
print(f"Length of test dataset: {len(c4_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in c4_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in c4_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in c4_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in c4_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(c4_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(c4_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 4. Quantization

### 4.1. BitsAndBytes

#### 4.1.1 BitsAndBytes 8-bit

In [5]:
# BNB Config 8-bit

from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "meta-llama/Meta-Llama-3-8B"

# BnB Quantization Configurations
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    load_in_4bit=False,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
    llm_int8_has_fp16_weight=False,
)

# Save path
import os
bnb_8bit_model_name = f"{model_name.split('/')[1]}-BNB-8"
bnb_8bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_8bit_model_name)
os.makedirs(bnb_8bit_model_path, exist_ok=True)

In [13]:
# Quantization 8-bit

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_8bit, 
    torch_dtype="auto",
    device_map=device
)
model_bnb_8bit.NAME = bnb_8bit_model_name

print(f"8-bit BNB Model Memory Footprint: {(model_bnb_8bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_8bit, bnb_8bit_model_path)

print(f"8-bit BnB model saved at: {bnb_8bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_8bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

8-bit BNB Model Memory Footprint: 8.46 GB
8-bit BnB model saved at: /nfs/students/daro/models/Meta-Llama-3-8B-BNB-8


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def quantize_and_save_model(model_name, bnb_config_8bit, device, save_path):
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
        
        # Load and quantize model
        model_bnb_8bit = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config_8bit,
            torch_dtype="auto",
            device_map=device
        )
        
        # Calculate and log memory footprint
        memory_footprint = model_bnb_8bit.get_memory_footprint() / (1024 ** 3)
        logger.info(f"8-bit BNB Model Memory Footprint: {memory_footprint:.2f} GB")
        
        # Save model and tokenizer
        model_bnb_8bit.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        
        logger.info(f"8-bit BnB model and tokenizer saved at: {save_path}")
        
        # Verify saved files
        saved_files = os.listdir(save_path)
        expected_files = ['config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json']
        for file in expected_files:
            if file not in saved_files:
                logger.warning(f"Expected file {file} not found in saved model directory.")
        
    except Exception as e:
        logger.error(f"Error during model quantization and saving: {str(e)}")
        raise

# Usage
device = "cuda" if torch.cuda.is_available() else "cpu"
save_path = "/nfs/students/daro/models/Meta-Llama-3-8B-BNB-8"

quantize_and_save_model(model_name, bnb_config_8bit, device, save_path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:__main__:8-bit BNB Model Memory Footprint: 8.46 GB
INFO:__main__:8-bit BnB model and tokenizer saved at: /nfs/students/daro/models/Meta-Llama-3-8B-BNB-8


#### 4.1.2 BitsAndBytes 4-bit

In [ ]:
# BNB Config 4-bit

import torch
from transformers import BitsAndBytesConfig

from src import MODEL_SAVE_PATH

# Define the device and model name
device = "cuda"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config_4bit = BitsAndBytesConfig(
    load_in_8bit=False,
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

# Save path
import os
bnb_4bit_model_name = f"{model_name.split('/')[1]}-bnb-4bit"
bnb_4bit_model_path = os.path.join(MODEL_SAVE_PATH, bnb_4bit_model_name)

In [ ]:
# Quantization 4-bit

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model_bnb_4bit = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config_4bit, 
    torch_dtype="auto",
    device_map=device
)
model_bnb_4bit.NAME = bnb_4bit_model_name

print(f"4-bit BNB Model Memory Footprint: {(model_bnb_4bit.get_memory_footprint() / (1024 ** 3)):.2f} GB")

from accelerate import Accelerator
accelerate = Accelerator()
accelerate.save_model(model_bnb_4bit, bnb_4bit_model_path)

print(f"4-bit BnB model saved at: {bnb_4bit_model_path}")

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(bnb_4bit_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

### 4.2 AWQ

In [7]:
from src import MODEL_SAVE_PATH
# AWQ Config
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# AWQ Calibration Split
awq_calib_split = "validation"

# Save path
import os
awq_model_name = f"{model_name.split('/')[1]}-AWQ-4-OASST"
awq_model_path = os.path.join(MODEL_SAVE_PATH, awq_model_name)

# Define the device and model name
device = "cuda"
model_name = "meta-llama/Meta-Llama-3-8B"

In [9]:
# # AWQ Quantization

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

awq_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# awq_model = AutoAWQForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map=device
# )

# # Quantize with wikitext validation as calibration data
# awq_model.quantize(
#     tokenizer=tokenizer,
#     quant_config=awq_config,
#     calib_data=oasst_dataloader,  # Pass the loaded validation dataset here
# )
awq_model.NAME = awq_model_name

# Save quantized model
awq_model.save_quantized(awq_model_path)
awq_tokenizer.save_pretrained(awq_model_path)

print(f'Model is quantized and saved at "{awq_model_path}"')

# from src.models.utils_llm import calculate_model_size
# calculate_model_size(awq_model_path)
# from src.models.utils_llm import print_gpu_utilization
# print_gpu_utilization()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model is quantized and saved at "/nfs/students/daro/models/Meta-Llama-3-8B-AWQ-4-OASST"


In [51]:
model.eval()
outputs = model(wikitext_dataloader.dataset[1][0].to(device).unsqueeze(0))

In [57]:
logits = outputs.logits
print(logits.shape)
argmax_indices = torch.argmax(logits, dim=2)
print(argmax_indices)
print(wikitext_dataloader.dataset[1][1])

torch.Size([1, 2048, 32000])
tensor([[ 573,  304,  278,  ..., 5492,  491, 4699]], device='cuda:0')
tensor([-100, -100, -100,  ..., -100, -100,  491])


In [7]:
import time

calib_text = []
for batch in wikitext_dataloader:
    input_ids, labels = batch
    decoded_text = tokenizer.decode(input_ids[0].tolist())
    calib_text.append(decoded_text)

In [8]:
for i, batch in enumerate(wikitext_dataloader):
    if i >= 4:
        break
    input_ids, labels = batch
    print(f"Batch {i+1}:")
    print(f"  Input IDs: {input_ids[0][:5]}")
    print(f"  Labels: {labels[0]}")

Batch 1:
  Input IDs: tensor([29906, 11791,  1723,  6483,   322])
  Labels: tensor([11791,  1723,  6483,   322,   591,  1141, 29879, 29871, 29896, 29941,
        23864,   869,   739,   338,   697,   310,   278,  1436,   342,  6455,
          310,   385,  7137, 29885,   687,   784,  2209,   284,  2343,   869,
          739,   471,  1476, 19214,   373,   967,  2625,   304,   278,  7062,
          310,   263, 13849,   284, 27446,   869,   450, 13849,   471, 10943,
          472,   263, 10809,   310, 29871, 29945, 17963,   313, 29871, 29896,
        29953, 11791,  1723,  2645,   263, 10710,  8328, 18994,   310,   278,
         3268,   297, 29871, 29896, 29929, 29953, 29947,  2056,   372,   756,
         1063, 29797,   304,   278, 11095,  4721,  1990,   293,   869,  2860,
        20699,   372,   471, 12919,   337,  8399,  1000,  2056,   372,   471,
         6153,   304,   278, 21037,   316,  5459,  1336, 14046,   316,  1060,
          284, 14274,   297, 29871, 29896, 29929, 29947, 29953,   

In [6]:
import time
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

calib_text = []
for batch in oasst_dataloader:
    input_ids, labels = batch
    decoded_text = tokenizer.decode(input_ids[0].tolist())
    calib_text.append(decoded_text)
    
awq_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

awq_model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map=device
)

start_time = time.time()
awq_model.quantize(
    tokenizer=tokenizer,
    quant_config=awq_config,
    calib_data=calib_text,
)
end_time = time.time()  # End time measurement
awq_model.QUANT_TIME = end_time - start_time

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

AWQ: 100%|██████████| 32/32 [19:03<00:00, 35.73s/it]


In [28]:
import torch
import logging
from typing import List, Union
from datasets import load_dataset


def get_calib_dataset(
    data: Union[str, List[str], List[List[int]]] = "pileval",
    tokenizer=None,
    n_samples=128,
    max_seq_len=512,
    split="train",
    text_column="text",
):
    if isinstance(data, str):
        if data == "pileval":
            dataset = load_dataset("mit-han-lab/pile-val-backup", split="validation")
        else:
            dataset = load_dataset(data, split=split)

        dataset = dataset.shuffle(seed=42)

    elif isinstance(data, list):
        if isinstance(data[0], str):
            dataset = [{text_column: text} for text in data]
        elif isinstance(data[0][0], int):
            dataset = data
        else:
            raise NotImplementedError(
                "Either pass a string to a huggingface dataset or a list"
                "that is preprocessed with one sample of text per element"
                " or a list of list of int for tokenized words."
            )
    else:
        raise NotImplementedError(
            "Either pass a string to a huggingface dataset or a list"
            "that is preprocessed with one sample of text per element"
            " or a list of list of int for tokenized words."
        )

    samples = []
    n_run = 0
    for data in dataset:
        if isinstance(data, list):
            line_encoded = data
        else:
            line = data[text_column]
            line = line.strip()
            line_encoded = tokenizer.encode(line)
        if len(line_encoded) > max_seq_len:
            continue
        sample = torch.tensor([line_encoded])
        if sample.numel() == 0:
            continue
        samples.append(sample)
        n_run += 1
        if n_run == n_samples:
            break
    # now concatenate all samples and split according to max sequence length
    cat_samples = torch.cat(samples, dim=1)
    n_split = cat_samples.shape[1] // max_seq_len
    logging.debug(f" * Split into {n_split} blocks")
    return [
        cat_samples[:, i * max_seq_len : (i + 1) * max_seq_len] for i in range(n_split)
    ]
    

calib_text = []
for batch in wikitext_dataloader:
    input_ids, labels = batch
    decoded_text = tokenizer.decode(input_ids[0].tolist())
    calib_text.append(decoded_text)  
get_calib_dataset(data=calib_text, tokenizer=tokenizer, n_samples=128, max_seq_len=2048, split="validation", text_column="text")

KeyboardInterrupt: 

In [27]:
calib_text

['16 @.@ 9 million , the remainder having been displaced or killed . During this time , Du Fu led a largely itinerant life unsettled by wars , associated famines and imperial displeasure . This period of unhappiness was the making of Du Fu as a poet : Even Shan Chou has written that , " What he saw around him — the lives of his family , neighbors , and strangers – what he heard , and what he hoped for or feared from the progress of various campaigns — these became the enduring themes of his poetry " . Even when he learned of the death of his youngest child , he turned to the suffering of others in his poetry instead of dwelling upon his own misfortunes . Du Fu wrote : \n\n\n Brooding on what I have lived through , if even I know such suffering , the common man must surely be rattled by the winds . \n\n\n In 756 , Emperor Xuanzong was forced to flee the capital and abdicate . Du Fu , who had been away from the city , took his family to a place of safety and attempted to join the court o

In [ ]:
# Load model and generate text
awq_model_path = "TinyLlama-1.1B-Chat-v1.0-awq"

awq_tokenizer = AutoTokenizer.from_pretrained(awq_model_path)
awq_model = AutoAWQForCausalLM.from_pretrained(
    awq_model_path,
    trust_remote_code=True,
)

# Generate text
prompt = "What is the weather like today?"
input_ids = awq_tokenizer.encode(prompt, return_tensors="pt")  # Convert text to tensors

output = awq_model.generate(input_ids)
generated_text = awq_tokenizer.decode(output[0], skip_special_tokens=True)

## 5. Evaluation

### 5.1. Perplexity

In [5]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()


################################
Evaluating Perplexity...
################################



In [6]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 21.9004
Perplexity: 4.49
Processing batch 1
Free GPU Memory (GB): 19.9434
Perplexity: 8.68
Processing batch 2
Free GPU Memory (GB): 19.9434
Perplexity: 9.55
Processing batch 3
Free GPU Memory (GB): 19.9434
Perplexity: 6.07
Processing batch 4
Free GPU Memory (GB): 19.9434
Perplexity: 5.45
Processing batch 5
Free GPU Memory (GB): 19.9434
Perplexity: 3.71
Processing batch 6
Free GPU Memory (GB): 19.9434
Perplexity: 3.78
Processing batch 7
Free GPU Memory (GB): 19.9434
Perplexity: 5.50
Processing batch 8
Free GPU Memory (GB): 19.9434
Perplexity: 7.66
Processing batch 9
Free GPU Memory (GB): 19.9434
Perplexity: 6.16
Processing batch 10
Free GPU Memory (GB): 19.9434
Perplexity: 5.65
Processing batch 11
Free GPU Memory (GB): 19.9434
Perplexity: 7.66
Processing batch 12
Free GPU Memory (GB): 19.9434
Perplexity: 7.04
Processing batch 13
Free GPU Memory (GB): 19.9434
Perplexity: 7.84
Processing batch 14
Free GPU Memo

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)